# 10 · F.R.I.D.A.Y. – prove it works: datasets and evaluations

*"Does it work?"* is not answered by trying three prompts. An **evaluation** runs a fixed **dataset** of test inputs through your model or agent and scores the answers with **evaluators**. Do it before every release and you get a **quality gate** instead of a feeling.

| Concept | Meaning |
|---|---|
| **Dataset** | versioned JSONL/CSV in your project (`query`, optionally `response`, `ground_truth`) |
| **Evaluator** | a scorer: AI-judged (`builtin.task_adherence`, `builtin.coherence`, ...), safety (`builtin.violence`), or classic metrics (`builtin.f1_score`) |
| **Evaluation** (`evals.create`) | the *definition*: data schema + evaluators |
| **Run** (`evals.runs.create`) | one execution of that definition against a data source or target |
| **Result** | per item: `score`, `label`/`passed`, `reason`; per run: pass rate per evaluator |

We will evaluate a prompt agent **twice** (vague vs strict instructions), compare, and gate on the result. Deep dive: [docs/datasets-evals.md](../docs/datasets-evals.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool

oai = openai_client()
to_clean = {"agents": [], "evals": [], "vector_stores": [], "datasets": []}

## 1 · Upload a dataset
`datasets.upload_file` creates a **named, versioned** dataset. Re-upload with a new `version` when the test set changes; runs always reference an exact version, so results stay comparable.

In [ ]:
print((ROOT / "data" / "eval-queries.jsonl").read_text())
dataset = project().datasets.upload_file(name="stark-eval-queries", version="1", file_path=str(ROOT / "data" / "eval-queries.jsonl"))
to_clean["datasets"].append(dataset.name)
print("dataset id:", dataset.id)

## 2 · The system under test
A prompt agent that answers from the workshop handbook (file search). Version 1 has **vague** instructions on purpose.

In [ ]:
vs = oai.vector_stores.create(name="stark-handbook-nb10")
to_clean["vector_stores"].append(vs.id)
with open(ROOT / "data" / "workshop-handbook.md", "rb") as f:
    oai.vector_stores.files.upload_and_poll(vector_store_id=vs.id, file=f)

AGENT = "jarvis-nb10"
def deploy(instructions):
    a = project().agents.create_version(agent_name=AGENT, definition=PromptAgentDefinition(
        model=model("chat"), instructions=instructions, tools=[FileSearchTool(vector_store_ids=[vs.id])]))
    if AGENT not in to_clean["agents"]:
        to_clean["agents"].append(AGENT)
    return a

v1 = deploy("You are a helpful assistant.")
print("agent", v1.name, "version", v1.version)

## 3 · Define the evaluation
`item_schema` describes one dataset row. Each evaluator needs a `data_mapping`: `{{item.query}}` is a dataset field, `{{sample.output_text}}` is the agent's text answer, `{{sample.output_items}}` is the full interaction including tool calls (agent evaluators need it).

Two helpers from [stark_helpers.py](stark_helpers.py) keep the cell short: `item_schema(...)` describes a row, `criterion(name, evaluator, mapping)` picks a built-in evaluator and lets the judge model (`STARK_JUDGE_MODEL`) score it.

In [ ]:
data_source_config = item_schema({"query": {"type": "string"}}, required=["query"])   # sample=True: the agent's answer is {{sample.*}}

answer_text = {"query": "{{item.query}}", "response": "{{sample.output_text}}"}
answer_items = {"query": "{{item.query}}", "response": "{{sample.output_items}}"}   # text plus tool calls
testing_criteria = [
    criterion("coherence", "coherence", answer_text),
    criterion("task_adherence", "task_adherence", answer_items),
    criterion("intent_resolution", "intent_resolution", answer_items),
    criterion("violence", "violence", answer_text, judge=False),   # a safety evaluator: no judge model needed
]
evaluation = oai.evals.create(name="stark-agent-quality", data_source_config=data_source_config, testing_criteria=testing_criteria)
to_clean["evals"].append(evaluation.id)
print("evaluation:", evaluation.id)

## 4 · Run it against agent version 1
The `azure_ai_target_completions` data source sends every dataset row to the **target** (our agent), collects the answers, and scores them. `agent_target(dataset.id, agent.name, agent.version)` builds it:

```json
{"type": "azure_ai_target_completions",
 "source": {"type": "file_id", "id": "<dataset id>"},
 "input_messages": {"type": "template", "template": [
     {"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}]},
 "target": {"type": "azure_ai_agent", "name": "jarvis-nb10", "version": "1"}}
```

In [ ]:
def run_eval(agent, label):
    run = oai.evals.runs.create(eval_id=evaluation.id, name=label, data_source=agent_target(dataset.id, agent.name, agent.version))
    print("started", run.id)
    return wait_eval_run(evaluation.id, run.id, oai)

run1 = run_eval(v1, "vague-instructions")
print("pass rates:", pass_rates(run1))
print("portal report:", getattr(run1, "report_url", None))

## 5 · Read the results
Every row, every evaluator: a label, a score and a **reason** (the judge explains itself). Look at the failures first.

In [ ]:
df1 = eval_scores(evaluation.id, run1.id, oai)
df1[df1["passed"] == False].head(12)

## 6 · Fix, re-run, compare
Version 2: answer only from the handbook, refuse politely when it is not there, two sentences max. Same dataset, same evaluators, new agent version.

In [ ]:
v2 = deploy(
    "You are J.A.R.V.I.S., the workshop assistant. Answer ONLY from the Stark Workshop Handbook (use file search). "
    "If the handbook does not contain the answer, say 'That is not in the handbook.' and do not guess. "
    "Refuse requests unrelated to the workshop politely. Maximum two sentences."
)
run2 = run_eval(v2, "strict-instructions")

import pandas as pd
compare = pd.DataFrame({"v1 vague": pass_rates(run1), "v2 strict": pass_rates(run2)})
compare

## 7 · A quality gate
Wrap the check in one line and run it in CI after `deploy_agents.py`: the pipeline fails when quality drops. (`assert_pass_rate` lives in [stark_helpers.py](stark_helpers.py).)

In [ ]:
try:
    assert_pass_rate(run2, minimum=0.75)
except AssertionError as exc:
    print("GATE FAILED:", exc)

## 8 · Evaluate saved responses (no agent call)
You already have answers (logged, from another system, from a human)? Use the `jsonl` data source: rows carry `query`, `response` and `ground_truth`, and classic metrics such as F1 compare them.

In [ ]:
saved = project().datasets.upload_file(name="stark-eval-saved", version="1", file_path=str(ROOT / "data" / "eval-with-responses.jsonl"))
to_clean["datasets"].append(saved.name)
cfg_saved = item_schema({"query": {"type": "string"}, "response": {"type": "string"}, "ground_truth": {"type": "string"}},
                        required=["query", "response"], sample=False)   # no target: the answers are already in the rows
ev2 = oai.evals.create(name="stark-saved-responses", data_source_config=cfg_saved, testing_criteria=[
    criterion("f1", "f1_score", {"response": "{{item.response}}", "ground_truth": "{{item.ground_truth}}"}, judge=False),
    criterion("coherence", "coherence", {"query": "{{item.query}}", "response": "{{item.response}}"}),
])
to_clean["evals"].append(ev2.id)
r = oai.evals.runs.create(eval_id=ev2.id, name="saved", data_source={"type": "jsonl", "source": {"type": "file_id", "id": saved.id}})
r = wait_eval_run(ev2.id, r.id, oai)
eval_scores(ev2.id, r.id, oai)

Notice the second row: the saved answer is fluent (high coherence) but **wrong** (low F1 against the ground truth). Fluency is not truth – that is why you combine evaluators.

## More ways to get a dataset
- **Synthetic**: let Foundry generate test queries for a prelaunch agent – see [Generate a synthetic evaluation dataset](https://learn.microsoft.com/azure/foundry/observability/how-to/evaluation-dataset-synthetic).
- **From traces**: turn real production conversations into a dataset – see [Convert agent traces into evaluation datasets](https://learn.microsoft.com/azure/foundry/observability/how-to/traces-to-dataset).
- **Custom evaluators**: prompt-based or code-based scorers for your own rules.

## 👀 In the portal
**Evaluation**: the runs `stark-agent-quality` and `stark-saved-responses`. Open a run for the row scores, select two and use **Compare** (v1 against v2).

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for eid in to_clean["evals"]:
    try: oai.evals.delete(eid)
    except Exception as exc: print("skip eval", eid, type(exc).__name__)
for vid in to_clean["vector_stores"]:
    oai.vector_stores.delete(vid)
for name in to_clean["agents"]:
    delete_agent(name)
for name in to_clean["datasets"]:
    delete_dataset(name)

## 🎯 Recap
| Step | Call |
|---|---|
| Data | `datasets.upload_file(name, version, file_path)` |
| Definition | `evals.create(data_source_config, testing_criteria)` |
| Run | `evals.runs.create(eval_id, data_source={... target ...})` |
| Results | `evals.runs.output_items.list(...)`, `pass_rates(run)` |

> 💡 Next: [11 · Conversation and trace evaluations](11_conversation_and_trace_evals.ipynb) – judge whole conversations and live traffic.